# 03 — Customer Segmentation

Build adult-only, unsupervised customer segments and connect them to the saved income classifier. The goal is a small set of interpretable, activateable customer groups plus segment × score recipes for marketing.

| Section | Purpose |
|---|---|
| [Initial plan](#initial-plan) | Map the segmentation plan to notebook chunks. |
| [Load + feature space](#load-features) | Recreate raw data, adult filter, deterministic cleaning, and segmentation features. |
| [Clustering matrix](#matrix) | Build the numeric/categorical preprocessing matrix. |
| [Method comparison](#method-comparison) | Compare scalable clustering methods and choose K. |
| [Stability](#stability) | Check selected segmentation stability on subsamples. |
| [Segment profiles](#profiles) | Profile, interpret, name, and optionally sub-segment. |
| [Classifier hand-off](#classifier-handoff) | Combine segments with income scores for marketing recipes. |
| [Save artifact](#save-artifact) | Save a reusable segmentation bundle and verify raw-row prediction. |

<a id="initial-plan"></a>

## Initial plan

| # | Segmentation requirement | Addressed in notebook |
|---:|---|---|
| 1 | Use adults only so children do not form a trivial non-worker cluster. | [Load + adult filter + segmentation feature space](#load-features) |
| 2 | Keep clustering unsupervised; use income only after segments form. | [Method comparison + K selection](#method-comparison), [Final segments: profile, interpret, name + sub-segmentation](#profiles), [Combine with the classifier: segment × score recipes](#classifier-handoff) |
| 3 | Use a compact, marketing-readable feature space and exclude sensitive attributes by default. | [Load + adult filter + segmentation feature space](#load-features), [Combine with the classifier: segment × score recipes](#classifier-handoff) |
| 4 | Compare common scalable methods and choose a small K with a 5% weighted activation floor. | [Method comparison + K selection](#method-comparison) |
| 5 | Validate segment stability with subsample ARI. | [Stability check](#stability) |
| 6 | Translate clusters into named, weighted population profiles. | [Final segments: profile, interpret, name + sub-segmentation](#profiles) |
| 7 | Combine segment labels with classifier scores for actionable targeting recipes. | [Combine with the classifier: segment × score recipes](#classifier-handoff) |
| 8 | Save a reusable bundle that predicts segment labels from raw rows. | [Save artifact + closing observations](#save-artifact) |

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import display

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("../data_raw")
COLUMNS_FILE = DATA_DIR / "census-bureau.columns"
DATA_FILE = DATA_DIR / "census-bureau.data"
MODEL_DIR = Path("../outputs/models")
CLASSIFIER_MODEL_PATH = MODEL_DIR / "income_classifier_bundle.joblib"
SEGMENTATION_MODEL_PATH = MODEL_DIR / "segmentation_bundle.joblib"
FIGURE_DIR = Path("../outputs/figures")
REPORT_DPI = 180

UNKNOWN_VALUE = "Unknown"
ADULT_AGE_THRESHOLD = 18
SILHOUETTE_SAMPLE_SIZE = 8_000
AGGLOMERATIVE_SAMPLE_SIZE = 10_000
ACTIVATION_FLOOR_WEIGHTED_PCT = 5.0

REPORT_PALETTE = {
    "blue": "#1F5A7A",
    "teal": "#3A8F8C",
    "orange": "#D9822B",
    "red": "#B94A48",
    "green": "#4F8A5B",
    "purple": "#6B5B95",
    "gray": "#6B7280",
    "light_gray": "#E5E7EB",
}
REPORT_LINE_COLORS = [REPORT_PALETTE[c] for c in ["blue", "orange", "teal", "purple", "green"]]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 120)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#D1D5DB",
    "axes.labelcolor": "#111827",
    "axes.titlecolor": "#111827",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "xtick.color": "#374151",
    "ytick.color": "#374151",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "legend.fontsize": 9,
    "legend.frameon": False,
    "grid.color": "#E5E7EB",
    "grid.linewidth": 0.8,
})


def polish_axes(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, alpha=0.8)
    return ax


def save_report_figure(fig, filename: str):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=REPORT_DPI, bbox_inches="tight", facecolor="white")
    return path

<a id="load-features"></a>

## Load + adult filter + segmentation feature space

This notebook is independent of the previous notebooks. It re-loads the raw Census data, recreates the target for segment profiling, filters to adults, and builds a compact feature set for clustering. The income label is not used as a clustering input.

In [2]:
with open(COLUMNS_FILE, "r") as f:
    columns = [line.strip() for line in f if line.strip()]

df_raw = pd.read_csv(DATA_FILE, header=None, names=columns, skipinitialspace=True)

assert df_raw.shape == (199_523, 42)
assert df_raw["label"].value_counts().to_dict() == {"- 50000.": 187_141, "50000+.": 12_382}

LABEL_MAP = {"- 50000.": 0, "50000+.": 1}
df = df_raw.copy()
df["over_50k"] = df["label"].map(LABEL_MAP)
assert df["over_50k"].notna().all()

weight_col = "weight"
target_col = "over_50k"
sensitive_attribute_cols = ["sex", "race", "hispanic origin", "citizenship"]

df_adults_raw = df.loc[df["age"] >= ADULT_AGE_THRESHOLD].copy()
raw_adults = df_raw.loc[df_adults_raw.index].copy()

adult_summary = pd.DataFrame([{
    "records_all": len(df),
    "records_adults": len(df_adults_raw),
    "adult_record_share_pct": len(df_adults_raw) / len(df) * 100,
    "weighted_population_all": df[weight_col].sum(),
    "weighted_population_adults": df_adults_raw[weight_col].sum(),
    "adult_weighted_share_pct": df_adults_raw[weight_col].sum() / df[weight_col].sum() * 100,
    "adult_positive_rate_pct": df_adults_raw[target_col].mean() * 100,
    "adult_weighted_positive_rate_pct": np.average(df_adults_raw[target_col], weights=df_adults_raw[weight_col]) * 100,
}])
adult_summary.round(3)

,records_all,records_adults,adult_record_share_pct,weighted_population_all,weighted_population_adults,adult_weighted_share_pct,adult_positive_rate_pct,adult_weighted_positive_rate_pct
0,199523,143531,71.937,3.472459e+08,2.535216e+08,73.009,8.625,8.772


In [3]:
categorical_segmentation_features = [
    "education",
    "major occupation code",
    "marital stat",
    "detailed household summary in household",
    "full or part time employment stat",
]

raw_numeric_segmentation_features = [
    "age",
    "capital gains",
    "dividends from stocks",
    "wage per hour",
    "weeks worked in year",
    "num persons worked for employer",
]

log1p_source_cols = ["capital gains", "dividends from stocks", "wage per hour"]
log1p_feature_map = {
    "capital gains": "log1p_capital_gains",
    "dividends from stocks": "log1p_dividends_from_stocks",
    "wage per hour": "log1p_wage_per_hour",
}

numeric_segmentation_features = [
    "age",
    "log1p_capital_gains",
    "log1p_dividends_from_stocks",
    "log1p_wage_per_hour",
    "weeks worked in year",
    "num persons worked for employer",
]

segmentation_features = categorical_segmentation_features + numeric_segmentation_features
assert not any(c in segmentation_features for c in sensitive_attribute_cols)

excluded_feature_notes = pd.DataFrame([
    {"excluded group": "Sensitive attributes", "examples": ", ".join(sensitive_attribute_cols), "reason": "Excluded by default; checked later as report-only sensitivity."},
    {"excluded group": "Migration artifacts", "examples": "migration code-change in msa/reg/move/sunbelt", "reason": "EDA showed they encode survey year availability rather than stable customer behavior."},
    {"excluded group": "Detailed classifier-only fields", "examples": "detailed industry/occupation, country of birth", "reason": "Excluded from default for interpretability; checked later as report-only expanded-feature sensitivity."},
])
excluded_feature_notes

,excluded group,examples,reason
0,Sensitive attributes,"sex, race, hispanic origin, citizenship",Excluded by default; checked later as report-only sensitivity.
1,Migration artifacts,migration code-change in msa/reg/move/sunbelt,EDA showed they encode survey year availability rather than stable customer behavior.
2,Detailed classifier-only fields,"detailed industry/occupation, country of birth",Excluded from default for interpretability; checked later as report-only expanded-feature sensitivity.


In [4]:
def clean_segmentation_frame(raw_df: pd.DataFrame, require_adults: bool = True) -> pd.DataFrame:
    out = raw_df.copy()
    if "over_50k" not in out.columns and "label" in out.columns:
        out["over_50k"] = out["label"].map(LABEL_MAP)
    if require_adults:
        if (out["age"] < ADULT_AGE_THRESHOLD).any():
            raise ValueError("Segmentation bundle expects adult rows only (age >= 18).")
        out = out.loc[out["age"] >= ADULT_AGE_THRESHOLD].copy()

    for col in categorical_segmentation_features + sensitive_attribute_cols:
        if col in out.columns:
            cleaned = out[col].astype("string").str.strip()
            cleaned = cleaned.replace({"?": UNKNOWN_VALUE, "NaN": UNKNOWN_VALUE, "Do not know": UNKNOWN_VALUE})
            cleaned = cleaned.fillna(UNKNOWN_VALUE)
            out[col] = cleaned.astype("string")

    for source, dest in log1p_feature_map.items():
        out[dest] = np.log1p(pd.to_numeric(out[source], errors="coerce").fillna(0))

    return out


df_seg = clean_segmentation_frame(df_adults_raw, require_adults=True)

assert len(df_seg) == 143_531
assert df_seg[categorical_segmentation_features].isna().sum().sum() == 0
assert df_seg[numeric_segmentation_features].isna().sum().sum() == 0
assert not any(c in segmentation_features for c in sensitive_attribute_cols)

feature_quality = pd.DataFrame({
    "feature": segmentation_features,
    "kind": ["categorical"] * len(categorical_segmentation_features) + ["numeric"] * len(numeric_segmentation_features),
    "missing_after_cleaning": [df_seg[c].isna().sum() for c in segmentation_features],
    "n_unique": [df_seg[c].nunique(dropna=False) for c in segmentation_features],
})
feature_quality

,feature,kind,missing_after_cleaning,n_unique
0,education,categorical,0,16
1,major occupation code,categorical,0,15
2,marital stat,categorical,0,7
3,detailed household summary in household,categorical,0,6
4,full or part time employment stat,categorical,0,8
5,age,numeric,0,73
6,log1p_capital_gains,numeric,0,132
7,log1p_dividends_from_stocks,numeric,0,1472
8,log1p_wage_per_hour,numeric,0,1235
9,weeks worked in year,numeric,0,53


### Observations

| # | Observation | Modeling implication |
|---:|---|---|
| 1 | Segmentation is restricted to adults only. | Avoids a trivial child/non-worker cluster and keeps the analysis aligned to retail activation. |
| 2 | Sensitive attributes are absent from the default segmentation feature list. | Segment definitions are based on education, work, household, and financial-behavior signals; sensitive attributes are checked later only as a sensitivity test. |
| 3 | The feature set is intentionally smaller than the classifier feature surface. | Segments should be interpretable, not just predictive. |

<a id="matrix"></a>

## Build the clustering matrix

The clustering matrix uses scaled numeric features and one-hot categorical features, then projects the encoded matrix to a compact representation for distance-based clustering. The projection is fit on the adult segmentation universe, since this is unsupervised and no label is used.

In [5]:
def fit_categorical_dtype_map(frame: pd.DataFrame, categorical_cols: list[str]) -> dict[str, pd.CategoricalDtype]:
    dtype_map = {}
    for col in categorical_cols:
        cats = pd.Series(frame[col].dropna().unique()).astype("string").tolist()
        if UNKNOWN_VALUE not in cats:
            cats.append(UNKNOWN_VALUE)
        dtype_map[col] = pd.CategoricalDtype(categories=cats, ordered=False)
    return dtype_map


def apply_categorical_dtype_map(frame: pd.DataFrame, dtype_map: dict[str, pd.CategoricalDtype]) -> pd.DataFrame:
    out = frame.copy()
    for col, dtype in dtype_map.items():
        values = out[col].astype("string").where(out[col].astype("string").isin(dtype.categories), UNKNOWN_VALUE)
        out[col] = values.astype(dtype)
    return out


def fit_segmentation_preprocessor(frame: pd.DataFrame, categorical_cols: list[str], numeric_cols: list[str]):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True), categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )
    encoded = preprocessor.fit_transform(frame[categorical_cols + numeric_cols])
    n_encoded_features = encoded.shape[1]
    if n_encoded_features > 20:
        projection = TruncatedSVD(n_components=20, random_state=RANDOM_STATE)
        matrix = projection.fit_transform(encoded)
        projection_name = "TruncatedSVD"
        explained_variance = float(projection.explained_variance_ratio_.sum())
    else:
        projection = None
        matrix = encoded.toarray() if sparse.issparse(encoded) else np.asarray(encoded)
        projection_name = "none"
        explained_variance = np.nan
    return preprocessor, projection, matrix, projection_name, explained_variance


def transform_segmentation_matrix(frame: pd.DataFrame, preprocessor, projection):
    encoded = preprocessor.transform(frame[categorical_segmentation_features + numeric_segmentation_features])
    if projection is not None:
        return projection.transform(encoded)
    return encoded.toarray() if sparse.issparse(encoded) else np.asarray(encoded)


categorical_dtype_map = fit_categorical_dtype_map(df_seg, categorical_segmentation_features)
df_seg_typed = apply_categorical_dtype_map(df_seg, categorical_dtype_map)

seg_preprocessor, seg_projection, X_seg, projection_name, projection_explained_variance = fit_segmentation_preprocessor(
    df_seg_typed,
    categorical_segmentation_features,
    numeric_segmentation_features,
)

matrix_summary = pd.DataFrame([{
    "adult_rows": len(df_seg_typed),
    "encoded_features": seg_preprocessor.transform(df_seg_typed[categorical_segmentation_features + numeric_segmentation_features]).shape[1],
    "cluster_matrix_features": X_seg.shape[1],
    "projection": projection_name,
    "projection_explained_variance_ratio": projection_explained_variance,
}])
matrix_summary.round(4)

,adult_rows,encoded_features,cluster_matrix_features,projection,projection_explained_variance_ratio
0,143531,58,20,TruncatedSVD,0.9144


### Observations

| # | Observation | Modeling implication |
|---:|---|---|
| 1 | The preprocessing pipeline is fit once on the adult segmentation universe. | This is acceptable for unsupervised clustering because no income label or future segment profile information is used. |
| 2 | Numeric and categorical preprocessing are stored as fitted objects. | The final bundle can assign new raw adult rows to the same segment space. |

<a id="method-comparison"></a>

## Method comparison + K selection

Compare common clustering methods with light defaults. Selection prioritizes structural quality and an activation floor; income separation is reported only after clusters form and used only as a tiebreaker among near-equal structural options.

In [6]:
def weighted_segment_summary(labels, y, weights) -> pd.DataFrame:
    labels = np.asarray(labels)
    y_arr = np.asarray(y)
    w_arr = np.asarray(weights)
    rows = []
    total_weight = w_arr.sum()
    total_positive_weight = w_arr[y_arr == 1].sum()
    for label in sorted(pd.unique(labels)):
        mask = labels == label
        seg_weight = w_arr[mask].sum()
        seg_positive_weight = w_arr[mask & (y_arr == 1)].sum()
        rows.append({
            "cluster": int(label),
            "rows": int(mask.sum()),
            "record_share_pct": mask.mean() * 100,
            "weighted_population": seg_weight,
            "weighted_share_pct": seg_weight / total_weight * 100,
            "weighted_positive_rate": seg_positive_weight / seg_weight if seg_weight else np.nan,
            "share_of_weighted_positives": seg_positive_weight / total_positive_weight if total_positive_weight else np.nan,
        })
    return pd.DataFrame(rows)


def income_separation_score(labels) -> float:
    profile = weighted_segment_summary(labels, df_seg_typed[target_col], df_seg_typed[weight_col])
    base = np.average(df_seg_typed[target_col], weights=df_seg_typed[weight_col])
    weights = profile["weighted_share_pct"] / 100
    return float(np.average((profile["weighted_positive_rate"] - base) ** 2, weights=weights))


def sample_indices(n: int, sample_size: int, seed: int = RANDOM_STATE) -> np.ndarray:
    rng = np.random.default_rng(seed)
    size = min(sample_size, n)
    return rng.choice(n, size=size, replace=False)


def fit_predict_candidate(method: str, k: int, X: np.ndarray, seed: int = RANDOM_STATE):
    if method == "KMeans":
        model = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels = model.fit_predict(X)
        return model, labels
    if method == "MiniBatchKMeans":
        model = MiniBatchKMeans(n_clusters=k, n_init=10, batch_size=4096, random_state=seed)
        labels = model.fit_predict(X)
        return model, labels
    if method == "GaussianMixture":
        model = GaussianMixture(n_components=k, covariance_type="diag", n_init=3, random_state=seed)
        labels = model.fit_predict(X)
        return model, labels
    raise ValueError(method)


sil_idx = sample_indices(len(df_seg_typed), SILHOUETTE_SAMPLE_SIZE)
ag_idx = sample_indices(len(df_seg_typed), AGGLOMERATIVE_SAMPLE_SIZE, seed=RANDOM_STATE + 100)

candidate_rows = []
candidate_models = {}
for method in ["KMeans", "MiniBatchKMeans", "GaussianMixture"]:
    for k in range(2, 7):
        model, labels = fit_predict_candidate(method, k, X_seg)
        profile = weighted_segment_summary(labels, df_seg_typed[target_col], df_seg_typed[weight_col])
        sil = silhouette_score(X_seg[sil_idx], labels[sil_idx])
        db = davies_bouldin_score(X_seg, labels)
        candidate_rows.append({
            "method": method,
            "k": k,
            "deployable": True,
            "silhouette": sil,
            "davies_bouldin": db,
            "structure_score": sil - 0.10 * db,
            "min_record_share_pct": profile["record_share_pct"].min(),
            "min_weighted_share_pct": profile["weighted_share_pct"].min(),
            "income_separation": income_separation_score(labels),
        })
        candidate_models[(method, k)] = {"model": model, "labels": labels}

# Ward agglomerative is a non-deployable sanity check on a smaller sample.
for k in range(2, 7):
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels_sub = model.fit_predict(X_seg[ag_idx])
    profile_sub = weighted_segment_summary(labels_sub, df_seg_typed.iloc[ag_idx][target_col], df_seg_typed.iloc[ag_idx][weight_col])
    sil = silhouette_score(X_seg[ag_idx], labels_sub)
    db = davies_bouldin_score(X_seg[ag_idx], labels_sub)
    candidate_rows.append({
        "method": "AgglomerativeWard",
        "k": k,
        "deployable": False,
        "silhouette": sil,
        "davies_bouldin": db,
        "structure_score": sil - 0.10 * db,
        "min_record_share_pct": profile_sub["record_share_pct"].min(),
        "min_weighted_share_pct": profile_sub["weighted_share_pct"].min(),
        "income_separation": np.nan,
    })

candidate_results = pd.DataFrame(candidate_rows)
eligible = candidate_results[
    (candidate_results["deployable"])
    & (candidate_results["min_weighted_share_pct"] >= ACTIVATION_FLOOR_WEIGHTED_PCT)
].copy()

best_score = eligible["structure_score"].max()
near_best = eligible[eligible["structure_score"] >= best_score - 0.01].copy()
near_best = near_best.sort_values(["k", "income_separation"], ascending=[True, False])
selected_row = near_best.iloc[0]
selected_method = selected_row["method"]
selected_k = int(selected_row["k"])
selected_model = candidate_models[(selected_method, selected_k)]["model"]
segment_labels = candidate_models[(selected_method, selected_k)]["labels"]

print("Selected segmentation:", selected_method, "k=", selected_k)
print("Selection structure score:", round(float(selected_row["structure_score"]), 4))
print("Near-best candidates within 0.01 structure score:")
display(near_best.sort_values("structure_score", ascending=False).round(4))

candidate_results.sort_values(["deployable", "structure_score"], ascending=[False, False]).round(4)

Selected segmentation: KMeans k= 4
Selection structure score: 0.1585
Near-best candidates within 0.01 structure score:


,method,k,deployable,silhouette,davies_bouldin,structure_score,min_record_share_pct,min_weighted_share_pct,income_separation
2,KMeans,4,True,0.2865,1.2797,0.1585,5.0804,5.0307,0.0053


,method,k,deployable,silhouette,davies_bouldin,structure_score,min_record_share_pct,min_weighted_share_pct,income_separation
2,KMeans,4,True,0.2865,1.2797,0.1585,5.0804,5.0307,0.0053
1,KMeans,3,True,0.2638,1.4410,0.1197,7.5984,7.7727,0.0029
4,KMeans,6,True,0.2485,1.3084,0.1177,5.0721,5.0249,0.0081
6,MiniBatchKMeans,3,True,0.2537,1.3911,0.1146,5.1034,5.0524,0.0050
11,GaussianMixture,3,True,0.2538,1.4467,0.1091,7.5991,7.7733,0.0026
3,KMeans,5,True,0.2299,1.3677,0.0932,5.0721,5.0260,0.0058
14,GaussianMixture,6,True,0.2263,1.3702,0.0893,5.1097,5.0595,0.0078
0,KMeans,2,True,0.2298,1.6151,0.0683,33.0904,32.5787,0.0025
12,GaussianMixture,4,True,0.2385,1.7018,0.0683,6.6125,6.7967,0.0072
5,MiniBatchKMeans,2,True,0.2298,1.6170,0.0681,33.1782,32.6691,0.0025


### Observations

| # | Observation | Modeling implication |
|---:|---|---|
| 1 | Model selection uses structural clustering metrics and a weighted activation floor. | The income label remains excluded from clustering selection. |
| 2 | Agglomerative Ward is a sanity check only. | The final model must support assigning future prospects without refitting. |

<a id="stability"></a>

## Stability check

Refit the selected method on two random 80% adult subsamples and compare cluster assignments on overlapping rows. ARI is label-permutation invariant, so it is appropriate for checking clustering stability.

In [7]:
def fit_full_segmentation_pipeline(frame: pd.DataFrame, method: str, k: int, seed: int = RANDOM_STATE):
    dtype_map = fit_categorical_dtype_map(frame, categorical_segmentation_features)
    typed = apply_categorical_dtype_map(frame, dtype_map)
    preprocessor, projection, matrix, projection_name, explained = fit_segmentation_preprocessor(
        typed,
        categorical_segmentation_features,
        numeric_segmentation_features,
    )
    model, labels = fit_predict_candidate(method, k, matrix, seed=seed)
    return {
        "dtype_map": dtype_map,
        "typed": typed,
        "preprocessor": preprocessor,
        "projection": projection,
        "projection_name": projection_name,
        "projection_explained_variance": explained,
        "matrix": matrix,
        "model": model,
        "labels": labels,
    }


def predict_with_pipeline(frame: pd.DataFrame, pipeline: dict) -> np.ndarray:
    typed = apply_categorical_dtype_map(frame, pipeline["dtype_map"])
    X = pipeline["preprocessor"].transform(typed[categorical_segmentation_features + numeric_segmentation_features])
    if pipeline["projection"] is not None:
        X = pipeline["projection"].transform(X)
    else:
        X = X.toarray() if sparse.issparse(X) else np.asarray(X)
    return pipeline["model"].predict(X)

rng_a = np.random.default_rng(RANDOM_STATE + 11)
rng_b = np.random.default_rng(RANDOM_STATE + 29)
idx_all = np.array(df_seg_typed.index)
sub_a_idx = rng_a.choice(idx_all, size=int(0.80 * len(idx_all)), replace=False)
sub_b_idx = rng_b.choice(idx_all, size=int(0.80 * len(idx_all)), replace=False)
overlap_idx = np.intersect1d(sub_a_idx, sub_b_idx)

pipeline_a = fit_full_segmentation_pipeline(df_seg_typed.loc[sub_a_idx], selected_method, selected_k, seed=RANDOM_STATE + 11)
pipeline_b = fit_full_segmentation_pipeline(df_seg_typed.loc[sub_b_idx], selected_method, selected_k, seed=RANDOM_STATE + 29)

overlap_frame = df_seg_typed.loc[overlap_idx]
labels_a = predict_with_pipeline(overlap_frame, pipeline_a)
labels_b = predict_with_pipeline(overlap_frame, pipeline_b)
stability_ari = adjusted_rand_score(labels_a, labels_b)

if stability_ari >= 0.8:
    stability_reading = "stable enough to recommend"
elif stability_ari >= 0.6:
    stability_reading = "usable, but flag in report"
else:
    stability_reading = "unstable; revisit K/features"

stability_summary = pd.DataFrame([{
    "selected_method": selected_method,
    "selected_k": selected_k,
    "subsample_a_rows": len(sub_a_idx),
    "subsample_b_rows": len(sub_b_idx),
    "overlap_rows": len(overlap_idx),
    "ari_on_overlap": stability_ari,
    "reading": stability_reading,
}])
stability_summary.round(4)

,selected_method,selected_k,subsample_a_rows,subsample_b_rows,overlap_rows,ari_on_overlap,reading
0,KMeans,4,114824,114824,91840,0.9997,stable enough to recommend


### Observations

| # | Observation | Modeling implication |
|---:|---|---|
| 1 | Stability is checked by refitting the complete preprocessing + clustering pipeline. | The ARI reflects both preprocessing and clustering variation, not only label assignment noise. |
| 2 | The final model is the selected method refit on all adult rows. | Segment profiles and artifacts use the same labels. |

<a id="profiles"></a>

## Final segments: profile, interpret, name + sub-segmentation

Profile each segment using weighted population statistics. Segment names are descriptive labels driven by the dominant pattern; they are not personas.

In [8]:
def weighted_mean(values, weights):
    return float(np.average(values, weights=weights))


def weighted_mode(values: pd.Series, weights: pd.Series):
    tmp = pd.DataFrame({"value": values.astype("string"), "weight": weights})
    grouped = tmp.groupby("value", dropna=False)["weight"].sum().sort_values(ascending=False)
    if grouped.empty:
        return UNKNOWN_VALUE, np.nan
    top = grouped.index[0]
    share = grouped.iloc[0] / grouped.sum()
    return str(top), float(share)


def build_segment_profile(frame: pd.DataFrame, labels: np.ndarray, names: dict[int, str] | None = None) -> pd.DataFrame:
    work = frame.copy()
    work["segment"] = labels
    total_weight = work[weight_col].sum()
    total_positive_weight = work.loc[work[target_col] == 1, weight_col].sum()
    rows = []
    for seg in sorted(work["segment"].unique()):
        seg_df = work.loc[work["segment"] == seg]
        seg_weight = seg_df[weight_col].sum()
        positive_weight = seg_df.loc[seg_df[target_col] == 1, weight_col].sum()
        top_occ, top_occ_share = weighted_mode(seg_df["major occupation code"], seg_df[weight_col])
        top_edu, top_edu_share = weighted_mode(seg_df["education"], seg_df[weight_col])
        top_marital, top_marital_share = weighted_mode(seg_df["marital stat"], seg_df[weight_col])
        top_household, top_household_share = weighted_mode(seg_df["detailed household summary in household"], seg_df[weight_col])
        top_work, top_work_share = weighted_mode(seg_df["full or part time employment stat"], seg_df[weight_col])
        rows.append({
            "segment": int(seg),
            "segment_name": names.get(int(seg), f"Segment {seg}") if names else f"Segment {seg}",
            "rows": len(seg_df),
            "weighted_size_pct": seg_weight / total_weight * 100,
            "in_segment_over_50k_pct": positive_weight / seg_weight * 100,
            "share_of_all_weighted_over_50k_pct": positive_weight / total_positive_weight * 100,
            "mean_age": weighted_mean(seg_df["age"], seg_df[weight_col]),
            "mean_weeks_worked": weighted_mean(seg_df["weeks worked in year"], seg_df[weight_col]),
            "mean_log1p_capital_gains": weighted_mean(seg_df["log1p_capital_gains"], seg_df[weight_col]),
            "mean_log1p_dividends": weighted_mean(seg_df["log1p_dividends_from_stocks"], seg_df[weight_col]),
            "top_occupation": top_occ,
            "top_occupation_weighted_share_pct": top_occ_share * 100,
            "top_education": top_edu,
            "top_education_weighted_share_pct": top_edu_share * 100,
            "top_marital_status": top_marital,
            "top_household_role": top_household,
            "top_work_status": top_work,
        })
    return pd.DataFrame(rows).sort_values("weighted_size_pct", ascending=False).reset_index(drop=True)


initial_profile = build_segment_profile(df_seg_typed, segment_labels)
initial_profile.round(2)

,segment,segment_name,rows,weighted_size_pct,in_segment_over_50k_pct,share_of_all_weighted_over_50k_pct,mean_age,mean_weeks_worked,mean_log1p_capital_gains,mean_log1p_dividends,top_occupation,top_occupation_weighted_share_pct,top_education,top_education_weighted_share_pct,top_marital_status,top_household_role,top_work_status
0,0,Segment 0,80307,56.32,11.30,72.54,38.32,46.39,0.00,0.80,Adm support including clerical,14.39,High school graduate,32.82,Married-civilian spouse present,Householder,Children or Armed Forces
1,1,Segment 1,45451,31.18,1.24,4.42,55.72,1.09,0.00,0.81,Not in universe,90.16,High school graduate,34.74,Married-civilian spouse present,Householder,Children or Armed Forces
2,3,Segment 3,10481,7.47,4.52,3.84,36.74,45.51,0.02,0.51,Adm support including clerical,19.32,High school graduate,40.52,Married-civilian spouse present,Householder,Children or Armed Forces
3,2,Segment 2,7292,5.03,33.47,19.20,48.58,39.50,8.70,1.80,Not in universe,21.88,High school graduate,27.04,Married-civilian spouse present,Householder,Children or Armed Forces


In [9]:
def propose_segment_names(profile: pd.DataFrame) -> dict[int, str]:
    names = {}
    used = set()
    for _, row in profile.iterrows():
        age = row["mean_age"]
        weeks = row["mean_weeks_worked"]
        pos_rate = row["in_segment_over_50k_pct"]
        size = row["weighted_size_pct"]
        cap_gain = row["mean_log1p_capital_gains"]
        dividends = row["mean_log1p_dividends"]
        occ = str(row["top_occupation"]).lower()

        if weeks < 10 and age >= 50:
            base = "Low-Workforce Adults"
        elif pos_rate >= 20 and (cap_gain >= 2 or dividends >= 1.5):
            base = "Investor Professionals"
        elif size >= 40 and weeks >= 35:
            base = "Steady Workers"
        elif weeks >= 35 and ("clerical" in occ or "sales" in occ or pos_rate < 8):
            base = "Service & Admin"
        elif weeks >= 35:
            base = "Working Adults"
        else:
            base = "Broad Adult Segment"

        name = base
        if name in used:
            name = f"{base} {int(row['segment'])}"
        used.add(name)
        names[int(row["segment"])] = name
    return names

segment_names = propose_segment_names(initial_profile)
segment_profile = build_segment_profile(df_seg_typed, segment_labels, segment_names)

segment_name_audit = segment_profile[[
    "segment", "segment_name", "weighted_size_pct", "in_segment_over_50k_pct",
    "mean_age", "mean_weeks_worked", "mean_log1p_capital_gains", "mean_log1p_dividends",
    "top_occupation", "top_education", "top_work_status"
]].copy()
segment_name_audit.round(2)

,segment,segment_name,weighted_size_pct,in_segment_over_50k_pct,mean_age,mean_weeks_worked,mean_log1p_capital_gains,mean_log1p_dividends,top_occupation,top_education,top_work_status
0,0,Steady Workers,56.32,11.30,38.32,46.39,0.00,0.80,Adm support including clerical,High school graduate,Children or Armed Forces
1,1,Low-Workforce Adults,31.18,1.24,55.72,1.09,0.00,0.81,Not in universe,High school graduate,Children or Armed Forces
2,3,Service & Admin,7.47,4.52,36.74,45.51,0.02,0.51,Adm support including clerical,High school graduate,Children or Armed Forces
3,2,Investor Professionals,5.03,33.47,48.58,39.50,8.70,1.80,Not in universe,High school graduate,Children or Armed Forces


In [10]:
def plot_segment_heatmap(frame: pd.DataFrame, labels: np.ndarray, names: dict[int, str]):
    work = frame.copy()
    work["segment"] = labels
    feature_rows = []
    base_weights = work[weight_col]
    for feature in numeric_segmentation_features:
        base_mean = weighted_mean(work[feature], base_weights)
        base_std = np.sqrt(np.average((work[feature] - base_mean) ** 2, weights=base_weights))
        row = {"feature": feature}
        for seg in sorted(work["segment"].unique()):
            seg_df = work.loc[work["segment"] == seg]
            seg_mean = weighted_mean(seg_df[feature], seg_df[weight_col])
            row[names[int(seg)]] = (seg_mean - base_mean) / base_std if base_std else 0
        feature_rows.append(row)
    heatmap_df = pd.DataFrame(feature_rows).set_index("feature")

    feature_labels = {
        "age": "Age",
        "log1p_capital_gains": "Capital gains",
        "log1p_dividends_from_stocks": "Dividends",
        "log1p_wage_per_hour": "Hourly wage",
        "weeks worked in year": "Weeks worked",
        "num persons worked for employer": "Employer count",
    }
    y_labels = [feature_labels.get(f, f) for f in heatmap_df.index]

    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    im = ax.imshow(heatmap_df.values, aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)
    ax.set_xticks(range(heatmap_df.shape[1]))
    ax.set_xticklabels(heatmap_df.columns, rotation=25, ha="right")
    ax.set_yticks(range(heatmap_df.shape[0]))
    ax.set_yticklabels(y_labels)
    ax.set_title("Segment profile vs. weighted adult average")
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = fig.colorbar(im, ax=ax, shrink=0.88)
    cbar.set_label("Weighted z-score")
    fig.tight_layout()
    path = save_report_figure(fig, "segmentation_segment_heatmap.png")
    plt.close(fig)
    return path, heatmap_df

heatmap_path, segment_heatmap_data = plot_segment_heatmap(df_seg_typed, segment_labels, segment_names)
print("Saved heatmap:", heatmap_path)
segment_heatmap_data.round(2)

Saved heatmap: ../outputs/figures/segmentation_segment_heatmap.png


,Steady Workers,Low-Workforce Adults,Investor Professionals,Service & Admin
feature,,,,
age,-0.33,0.66,0.25,-0.42
log1p_capital_gains,-0.23,-0.23,4.30,-0.22
log1p_dividends_from_stocks,-0.02,-0.01,0.45,-0.15
log1p_wage_per_hour,-0.29,-0.29,-0.06,3.43
weeks worked in year,0.62,-1.31,0.33,0.58
num persons worked for employer,0.49,-1.05,0.22,0.56


In [11]:
TSNE_SAMPLE_SIZE = 5_000
rng = np.random.default_rng(RANDOM_STATE)
tsne_idx = rng.choice(len(df_seg_typed), size=min(TSNE_SAMPLE_SIZE, len(df_seg_typed)), replace=False)

# t-SNE is for visualization only; model selection uses structural metrics above.
tsne_embedding = TSNE(
    n_components=2,
    perplexity=35,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE,
    max_iter=1_000,
).fit_transform(X_seg[tsne_idx])

tsne_df = pd.DataFrame({
    "tsne_1": tsne_embedding[:, 0],
    "tsne_2": tsne_embedding[:, 1],
    "segment": segment_labels[tsne_idx],
    "segment_name": [segment_names[int(s)] for s in segment_labels[tsne_idx]],
    "over_50k": df_seg_typed.iloc[tsne_idx][target_col].to_numpy(),
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharex=True, sharey=True)

segment_order = segment_profile["segment_name"].tolist()
color_map = dict(zip(segment_order, REPORT_LINE_COLORS))
for segment_name in segment_order:
    sub = tsne_df.loc[tsne_df["segment_name"] == segment_name]
    axes[0].scatter(sub["tsne_1"], sub["tsne_2"], s=8, alpha=0.68, color=color_map[segment_name], label=segment_name, linewidths=0)
axes[0].set_title("Colored by segment")
axes[0].legend(loc="upper left", bbox_to_anchor=(0.0, -0.08), ncol=2)

low = tsne_df.loc[tsne_df["over_50k"] == 0]
high = tsne_df.loc[tsne_df["over_50k"] == 1]
axes[1].scatter(low["tsne_1"], low["tsne_2"], s=7, alpha=0.22, color="#A7AEB8", label="<= $50K", linewidths=0)
axes[1].scatter(high["tsne_1"], high["tsne_2"], s=13, alpha=0.82, color=REPORT_PALETTE["red"], label="> $50K", linewidths=0)
axes[1].set_title("Same sample with income overlay")
axes[1].legend(loc="upper left", bbox_to_anchor=(0.0, -0.08), ncol=2)

for ax in axes:
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Segment structure in a 5,000-row adult sample", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
tsne_path = save_report_figure(fig, "segmentation_tsne_clusters.png")
plt.close(fig)

print("Saved t-SNE figure:", tsne_path)
tsne_df.groupby("segment_name", observed=True).size().rename("sample_rows").reset_index()

Saved t-SNE figure: ../outputs/figures/segmentation_tsne_clusters.png


,segment_name,sample_rows
0,Investor Professionals,252
1,Low-Workforce Adults,1555
2,Service & Admin,322
3,Steady Workers,2871


In [12]:
large_segments = segment_profile.loc[segment_profile["weighted_size_pct"] >= 50, "segment"].tolist()
subsegment_results = {}

for parent_segment in large_segments:
    parent_mask = segment_labels == parent_segment
    parent_df = df_seg_typed.loc[parent_mask].copy()
    parent_weight_total = parent_df[weight_col].sum()
    parent_pipeline_rows = []
    parent_fit = fit_full_segmentation_pipeline(parent_df, "KMeans", 2, seed=RANDOM_STATE)
    parent_X = parent_fit["matrix"]
    parent_sil_idx = sample_indices(len(parent_df), min(SILHOUETTE_SAMPLE_SIZE, len(parent_df)), seed=RANDOM_STATE + int(parent_segment))
    parent_candidates = []
    parent_models = {}
    for k in range(2, 6):
        model, labels = fit_predict_candidate("KMeans", k, parent_X, seed=RANDOM_STATE)
        profile = weighted_segment_summary(labels, parent_df[target_col], parent_df[weight_col])
        sil = silhouette_score(parent_X[parent_sil_idx], labels[parent_sil_idx])
        db = davies_bouldin_score(parent_X, labels)
        parent_candidates.append({
            "parent_segment": int(parent_segment),
            "parent_segment_name": segment_names[int(parent_segment)],
            "k": k,
            "silhouette": sil,
            "davies_bouldin": db,
            "structure_score": sil - 0.10 * db,
            "min_parent_weighted_share_pct": profile["weighted_population"].min() / parent_weight_total * 100,
        })
        parent_models[k] = {"model": model, "labels": labels, "profile": profile}
    parent_candidate_df = pd.DataFrame(parent_candidates)
    parent_eligible = parent_candidate_df[parent_candidate_df["min_parent_weighted_share_pct"] >= ACTIVATION_FLOOR_WEIGHTED_PCT]
    if len(parent_eligible):
        parent_best_score = parent_eligible["structure_score"].max()
        parent_near = parent_eligible[parent_eligible["structure_score"] >= parent_best_score - 0.01].sort_values("k")
        parent_selected_k = int(parent_near.iloc[0]["k"])
        parent_labels = parent_models[parent_selected_k]["labels"]
        parent_names = {int(i): f"{segment_names[int(parent_segment)]} subsegment {i}" for i in sorted(pd.unique(parent_labels))}
        parent_profile = build_segment_profile(parent_df, parent_labels, parent_names)
        subsegment_results[int(parent_segment)] = {
            "parent_segment_name": segment_names[int(parent_segment)],
            "selected_k": parent_selected_k,
            "candidate_results": parent_candidate_df,
            "profile": parent_profile,
        }

if subsegment_results:
    for parent, result in subsegment_results.items():
        print(f"Parent segment {parent}: {result['parent_segment_name']}, selected k={result['selected_k']}")
        display(result["profile"].round(2))
else:
    print("No segment exceeded the 50% weighted-size threshold; sub-segmentation skipped.")

Parent segment 0: Steady Workers, selected k=2


,segment,segment_name,rows,weighted_size_pct,in_segment_over_50k_pct,share_of_all_weighted_over_50k_pct,mean_age,mean_weeks_worked,mean_log1p_capital_gains,mean_log1p_dividends,top_occupation,top_occupation_weighted_share_pct,top_education,top_education_weighted_share_pct,top_marital_status,top_household_role,top_work_status
0,1,Steady Workers subsegment 1,69785,86.96,7.93,61.03,37.42,46.02,0.0,0.04,Adm support including clerical,14.61,High school graduate,34.78,Married-civilian spouse present,Householder,Children or Armed Forces
1,0,Steady Workers subsegment 0,10522,13.04,33.76,38.97,44.36,48.89,0.0,5.85,Professional specialty,26.31,Bachelors degree(BA AB BS),31.61,Married-civilian spouse present,Householder,Children or Armed Forces


### Observations

| # | Observation | Marketing implication |
|---:|---|---|
| 1 | Segment names are generated from weighted profiles, then audited in the table. | Names should be treated as descriptive shorthand, not demographic stereotypes. |
| 2 | Sub-segmentation is conditional on a segment being too large to activate as one audience. | The notebook avoids creating small, hard-to-use groups unless a parent segment needs more detail. |

<a id="classifier-handoff"></a>

## Combine with the classifier: segment × score recipes

Load the saved income classifier, score adult rows, and combine segment membership with top score bands. These recipes are the bridge from unsupervised segments to marketing activation.

In [13]:
def clean_for_classifier(raw_df: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    out = raw_df.copy()
    config = bundle["preprocessing_config"]
    unknown = config.get("unknown_value", UNKNOWN_VALUE)
    if "over_50k" not in out.columns and "label" in out.columns:
        out["over_50k"] = out["label"].map(LABEL_MAP)
    for col in bundle["categorical_dtype_map"]:
        cleaned = out[col].astype("string").str.strip()
        cleaned = cleaned.replace({"?": unknown, "NaN": unknown, "Do not know": unknown})
        cleaned = cleaned.fillna(unknown)
        dtype = bundle["categorical_dtype_map"][col]
        cleaned = cleaned.where(cleaned.isin(dtype.categories), unknown)
        out[col] = cleaned.astype(dtype)
    topcode = config.get("topcode_thresholds", {}).get("capital gains", 99_999)
    out["capital_gains_at_cap"] = (pd.to_numeric(out["capital gains"], errors="coerce") == topcode).astype(int)
    return out[bundle["feature_cols"]]


def to_classifier_ordinal_features(X: pd.DataFrame, categorical_cols: list[str]) -> pd.DataFrame:
    out = X.copy()
    for col in categorical_cols:
        out[col] = out[col].cat.codes.astype("int64")
    return out


def predict_income_from_raw(raw_df: pd.DataFrame, bundle: dict) -> np.ndarray:
    X_base = clean_for_classifier(raw_df, bundle)
    representation = bundle["feature_representation"]
    if representation == "ordinal":
        X_model = to_classifier_ordinal_features(X_base, [c for c in bundle["categorical_cols"] if c in X_base.columns])
        return bundle["model"].predict_proba(X_model)[:, 1]
    return bundle["model"].predict_proba(X_base)[:, 1]


classifier_bundle = joblib.load(CLASSIFIER_MODEL_PATH)
adult_income_scores = predict_income_from_raw(raw_adults, classifier_bundle)

df_scored = df_seg_typed.copy()
df_scored["segment"] = segment_labels
df_scored["segment_name"] = df_scored["segment"].map(segment_names)
df_scored["income_score"] = adult_income_scores

score_distribution = (
    df_scored.groupby("segment_name", observed=True)["income_score"]
    .quantile([0.10, 0.50, 0.90, 0.95])
    .unstack()
    .rename(columns={0.10: "score_p10", 0.50: "score_p50", 0.90: "score_p90", 0.95: "score_p95"})
    .reset_index()
)
score_distribution.round(4)

,segment_name,score_p10,score_p50,score_p90,score_p95
0,Investor Professionals,0.0030,0.1072,0.9439,0.9709
1,Low-Workforce Adults,0.0006,0.0031,0.0252,0.0488
2,Service & Admin,0.0011,0.0083,0.1246,0.2731
3,Steady Workers,0.0021,0.0246,0.3562,0.5840


In [14]:
def recipe_table(scored: pd.DataFrame, top_pcts=(5, 10, 20)) -> pd.DataFrame:
    rows = []
    population_base = np.average(scored[target_col], weights=scored[weight_col])
    for segment_name, seg_df in scored.groupby("segment_name", observed=True):
        segment_base = np.average(seg_df[target_col], weights=seg_df[weight_col])
        seg_sorted = seg_df.sort_values("income_score", ascending=False)
        for pct in top_pcts:
            n = max(1, int(np.ceil(len(seg_sorted) * pct / 100)))
            top_df = seg_sorted.head(n)
            weighted_reach = top_df[weight_col].sum()
            precision = np.average(top_df[target_col], weights=top_df[weight_col])
            rows.append({
                "segment_name": segment_name,
                "within_segment_top_pct": pct,
                "rows_contacted": len(top_df),
                "weighted_reach": weighted_reach,
                "weighted_reach_pct_of_adults": weighted_reach / scored[weight_col].sum() * 100,
                "recipe_over_50k_pct": precision * 100,
                "segment_baseline_over_50k_pct": segment_base * 100,
                "lift_vs_segment_baseline": precision / segment_base if segment_base else np.nan,
                "lift_vs_adult_population": precision / population_base if population_base else np.nan,
            })
    return pd.DataFrame(rows)

segment_score_recipes = recipe_table(df_scored)
recommended_recipes = (
    segment_score_recipes
    .sort_values(["recipe_over_50k_pct", "weighted_reach_pct_of_adults"], ascending=[False, False])
    .head(8)
)

plot_df = segment_score_recipes.loc[segment_score_recipes["within_segment_top_pct"] == 10].sort_values("recipe_over_50k_pct")
adult_base = np.average(df_scored[target_col], weights=df_scored[weight_col]) * 100
fig, ax = plt.subplots(figsize=(9.8, 4.9))
ax.barh(plot_df["segment_name"], plot_df["recipe_over_50k_pct"], color=REPORT_PALETTE["blue"])
ax.axvline(adult_base, color=REPORT_PALETTE["gray"], linestyle="--", label="Adult weighted baseline")
ax.set_xlabel("High-income share in top 10% score slice (%)")
ax.set_title("Score ranking finds useful pockets inside each segment")
for _, row in plot_df.iterrows():
    ax.text(row["recipe_over_50k_pct"] + 1.2, row["segment_name"], f'{row["recipe_over_50k_pct"]:.1f}%',
            va="center", ha="left", fontsize=9)
ax.set_xlim(0, max(100, plot_df["recipe_over_50k_pct"].max() + 8))
ax.legend(loc="lower right")
polish_axes(ax, grid_axis="x")
fig.tight_layout()
recipe_chart_path = save_report_figure(fig, "segmentation_score_recipes.png")
plt.close(fig)

print("Saved recipe chart:", recipe_chart_path)
display(segment_score_recipes.round(2))
print("Top recipes by precision")
display(recommended_recipes.round(2))

Saved recipe chart: ../outputs/figures/segmentation_score_recipes.png


,segment_name,within_segment_top_pct,rows_contacted,weighted_reach,weighted_reach_pct_of_adults,recipe_over_50k_pct,segment_baseline_over_50k_pct,lift_vs_segment_baseline,lift_vs_adult_population
0,Investor Professionals,5,365,657417.53,0.26,99.19,33.47,2.96,11.31
1,Investor Professionals,10,730,1309788.55,0.52,98.79,33.47,2.95,11.26
2,Investor Professionals,20,1459,2611564.42,1.03,96.61,33.47,2.89,11.01
3,Low-Workforce Adults,5,2273,4027904.59,1.59,15.33,1.24,12.33,1.75
4,Low-Workforce Adults,10,4546,7943710.02,3.13,9.43,1.24,7.58,1.07
5,Low-Workforce Adults,20,9091,15801809.59,6.23,5.35,1.24,4.31,0.61
6,Service & Admin,5,525,915456.51,0.36,55.99,4.52,12.40,6.38
7,Service & Admin,10,1049,1836725.63,0.72,36.73,4.52,8.13,4.19
8,Service & Admin,20,2097,3653358.45,1.44,21.03,4.52,4.66,2.40
9,Steady Workers,5,4016,7138247.95,2.82,82.86,11.30,7.33,9.45


Top recipes by precision


,segment_name,within_segment_top_pct,rows_contacted,weighted_reach,weighted_reach_pct_of_adults,recipe_over_50k_pct,segment_baseline_over_50k_pct,lift_vs_segment_baseline,lift_vs_adult_population
0,Investor Professionals,5,365,657417.53,0.26,99.19,33.47,2.96,11.31
1,Investor Professionals,10,730,1309788.55,0.52,98.79,33.47,2.95,11.26
2,Investor Professionals,20,1459,2611564.42,1.03,96.61,33.47,2.89,11.01
9,Steady Workers,5,4016,7138247.95,2.82,82.86,11.30,7.33,9.45
10,Steady Workers,10,8031,14340088.35,5.66,66.67,11.30,5.90,7.60
6,Service & Admin,5,525,915456.51,0.36,55.99,4.52,12.40,6.38
11,Steady Workers,20,16062,28467937.31,11.23,46.26,11.30,4.09,5.27
7,Service & Admin,10,1049,1836725.63,0.72,36.73,4.52,8.13,4.19


In [15]:
sensitive_categorical_features = categorical_segmentation_features + sensitive_attribute_cols
sensitive_df = clean_segmentation_frame(df_adults_raw, require_adults=True)

sensitive_dtype_map = fit_categorical_dtype_map(sensitive_df, sensitive_categorical_features)
sensitive_typed = apply_categorical_dtype_map(sensitive_df, sensitive_dtype_map)

sensitive_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_segmentation_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True), sensitive_categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)
sensitive_encoded = sensitive_preprocessor.fit_transform(sensitive_typed[sensitive_categorical_features + numeric_segmentation_features])
if sensitive_encoded.shape[1] > 20:
    sensitive_projection = TruncatedSVD(n_components=20, random_state=RANDOM_STATE)
    sensitive_X = sensitive_projection.fit_transform(sensitive_encoded)
else:
    sensitive_projection = None
    sensitive_X = sensitive_encoded.toarray() if sparse.issparse(sensitive_encoded) else np.asarray(sensitive_encoded)

sensitive_model, sensitive_labels = fit_predict_candidate(selected_method, selected_k, sensitive_X, seed=RANDOM_STATE)
sensitive_ari = adjusted_rand_score(segment_labels, sensitive_labels)

sensitive_reading = "segments materially similar; exclusion remains safer default" if sensitive_ari >= 0.8 else "segments diverge; treat as explicit policy trade-off"
sensitive_summary = pd.DataFrame([{
    "selected_method": selected_method,
    "selected_k": selected_k,
    "sensitive_features_added": ", ".join(sensitive_attribute_cols),
    "ari_vs_default_segments": sensitive_ari,
    "reading": sensitive_reading,
}])
sensitive_summary.round(4)

,selected_method,selected_k,sensitive_features_added,ari_vs_default_segments,reading
0,KMeans,4,"sex, race, hispanic origin, citizenship",0.9884,segments materially similar; exclusion remains safer default


In [16]:
expanded_categorical_features = categorical_segmentation_features + [
    "detailed industry recode",
    "detailed occupation recode",
    "country of birth father",
    "country of birth mother",
    "country of birth self",
]

expanded_df = clean_segmentation_frame(df_adults_raw, require_adults=True)
for col in expanded_categorical_features:
    cleaned = expanded_df[col].astype("string").str.strip()
    cleaned = cleaned.replace({"?": UNKNOWN_VALUE, "NaN": UNKNOWN_VALUE, "Do not know": UNKNOWN_VALUE})
    cleaned = cleaned.fillna(UNKNOWN_VALUE)
    expanded_df[col] = cleaned.astype("string")

expanded_dtype_map = fit_categorical_dtype_map(expanded_df, expanded_categorical_features)
expanded_typed = apply_categorical_dtype_map(expanded_df, expanded_dtype_map)

expanded_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_segmentation_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True), expanded_categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)
expanded_encoded = expanded_preprocessor.fit_transform(expanded_typed[expanded_categorical_features + numeric_segmentation_features])
if expanded_encoded.shape[1] > 20:
    expanded_projection = TruncatedSVD(n_components=20, random_state=RANDOM_STATE)
    expanded_X = expanded_projection.fit_transform(expanded_encoded)
else:
    expanded_projection = None
    expanded_X = expanded_encoded.toarray() if sparse.issparse(expanded_encoded) else np.asarray(expanded_encoded)

expanded_model, expanded_labels = fit_predict_candidate(selected_method, selected_k, expanded_X, seed=RANDOM_STATE)
expanded_ari = adjusted_rand_score(segment_labels, expanded_labels)
expanded_profile = build_segment_profile(expanded_typed, expanded_labels)

expanded_reading = (
    "expanded detailed fields leave the broad segment structure materially similar"
    if expanded_ari >= 0.8
    else "expanded detailed fields materially change the segments; inspect before keeping the default feature set"
)

expanded_feature_summary = pd.DataFrame([{
    "selected_method": selected_method,
    "selected_k": selected_k,
    "expanded_features_added": ", ".join([c for c in expanded_categorical_features if c not in categorical_segmentation_features]),
    "encoded_features_default": seg_preprocessor.transform(df_seg_typed[categorical_segmentation_features + numeric_segmentation_features]).shape[1],
    "encoded_features_expanded": expanded_encoded.shape[1],
    "ari_vs_default_segments": expanded_ari,
    "reading": expanded_reading,
}])

print("Expanded-feature sensitivity check")
display(expanded_feature_summary.round(4))
print("Expanded-feature profile, unnamed clusters")
display(expanded_profile[[
    "segment", "weighted_size_pct", "in_segment_over_50k_pct", "share_of_all_weighted_over_50k_pct",
    "mean_age", "mean_weeks_worked", "top_occupation", "top_education"
]].round(2))

Expanded-feature sensitivity check


,selected_method,selected_k,expanded_features_added,encoded_features_default,encoded_features_expanded,ari_vs_default_segments,reading
0,KMeans,4,"detailed industry recode, detailed occupation recode, country of birth father, country of birth mother, country of b...",58,281,0.9463,expanded detailed fields leave the broad segment structure materially similar


Expanded-feature profile, unnamed clusters


,segment,weighted_size_pct,in_segment_over_50k_pct,share_of_all_weighted_over_50k_pct,mean_age,mean_weeks_worked,top_occupation,top_education
0,2,56.34,11.29,72.53,38.38,46.28,Adm support including clerical,High school graduate
1,0,31.16,1.25,4.43,55.64,1.25,Not in universe,High school graduate
2,1,7.47,4.52,3.84,36.74,45.51,Adm support including clerical,High school graduate
3,3,5.02,33.52,19.20,48.56,39.55,Not in universe,High school graduate


### Observations

| # | Observation | Marketing implication |
|---:|---|---|
| 1 | Segment labels and classifier scores answer different questions. | Use segments for audience/message framing and scores for within-segment prioritization. |
| 2 | Sensitive attributes are checked only after default segments are formed. | The recommended default remains sensitive-attribute exclusion unless the business and legal case says otherwise. |
| 3 | Detailed industry/occupation and country-of-birth fields are tested as an expanded-feature sensitivity check. | If ARI is high, the default broad feature set preserves interpretability without losing the main segment structure. |

<a id="save-artifact"></a>

## Save artifact + closing observations

Save the fitted segmentation pipeline and verify it can assign raw adult rows to the same labels produced in the notebook.

In [17]:
def predict_segment_from_raw(raw_df: pd.DataFrame, bundle: dict) -> np.ndarray:
    cleaned = clean_segmentation_frame(raw_df, require_adults=True)
    typed = apply_categorical_dtype_map(cleaned, bundle["categorical_dtype_map"])
    X = bundle["preprocessor"].transform(typed[bundle["categorical_cols"] + bundle["numeric_cols"]])
    if bundle["projection"] is not None:
        X = bundle["projection"].transform(X)
    else:
        X = X.toarray() if sparse.issparse(X) else np.asarray(X)
    return bundle["model"].predict(X)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

segmentation_bundle = {
    "model": selected_model,
    "model_name": selected_method,
    "k": selected_k,
    "feature_cols": segmentation_features,
    "categorical_cols": categorical_segmentation_features,
    "numeric_cols": numeric_segmentation_features,
    "categorical_dtype_map": categorical_dtype_map,
    "preprocessor": seg_preprocessor,
    "projection": seg_projection,
    "preprocessing_config": {
        "unknown_value": UNKNOWN_VALUE,
        "sentinel_to_unknown": {"categorical_columns": ["?", "NaN", "Do not know"]},
        "log1p_columns": log1p_source_cols,
        "log1p_feature_map": log1p_feature_map,
        "adult_age_threshold": ADULT_AGE_THRESHOLD,
        "sensitive_attribute_cols_excluded": sensitive_attribute_cols,
        "activation_floor_weighted_pct": ACTIVATION_FLOOR_WEIGHTED_PCT,
    },
    "segment_names": segment_names,
    "segment_profile_table": segment_profile,
    "segment_score_recipes": segment_score_recipes,
    "subsegment_results": subsegment_results,
    "stability": {
        "ari_subsample": float(stability_ari),
        "ari_sensitive_check": float(sensitive_ari),
        "ari_expanded_feature_check": float(expanded_ari),
    },
    "benchmark": {
        "candidate_results": candidate_results.to_dict(orient="records"),
        "selected_method": selected_method,
        "selected_k": selected_k,
    },
    "metadata": {
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "random_state": RANDOM_STATE,
        "n_adults": int(len(df_seg_typed)),
        "weighted_population_adults": float(df_seg_typed[weight_col].sum()),
        "projection": projection_name,
        "projection_explained_variance_ratio": None if np.isnan(projection_explained_variance) else float(projection_explained_variance),
        "expanded_feature_sensitivity": expanded_feature_summary.to_dict(orient="records"),
        "sklearn_version": sklearn.__version__,
        "pandas_version": pd.__version__,
        "python_version": sys.version,
    },
}

joblib.dump(segmentation_bundle, SEGMENTATION_MODEL_PATH)
loaded_segmentation_bundle = joblib.load(SEGMENTATION_MODEL_PATH)

sample_raw = raw_adults.head(5)
roundtrip_segments = predict_segment_from_raw(sample_raw, loaded_segmentation_bundle)
notebook_segments = segment_labels[:5]
assert np.array_equal(roundtrip_segments, notebook_segments)

print("Saved segmentation bundle:", SEGMENTATION_MODEL_PATH)
print("Round-trip segments:", roundtrip_segments.tolist())
print("Segment names:", {int(k): v for k, v in segment_names.items()})
print("Bundle keys:", sorted(loaded_segmentation_bundle.keys()))

Saved segmentation bundle: ../outputs/models/segmentation_bundle.joblib
Round-trip segments: [1, 0, 1, 3, 2]
Segment names: {0: 'Steady Workers', 1: 'Low-Workforce Adults', 3: 'Service & Admin', 2: 'Investor Professionals'}
Bundle keys: ['benchmark', 'categorical_cols', 'categorical_dtype_map', 'feature_cols', 'k', 'metadata', 'model', 'model_name', 'numeric_cols', 'preprocessing_config', 'preprocessor', 'projection', 'segment_names', 'segment_profile_table', 'segment_score_recipes', 'stability', 'subsegment_results']


### Closing observations

| # | Observation | Report-ready implication |
|---:|---|---|
| 1 | Segments are adult-only, unsupervised, and profiled with population weights. | Section 4 can now describe customer groups without implying income was used to create them. |
| 2 | Classifier scores are layered on top of segments. | Marketing can choose both message framing (segment) and priority/reach (score band). |
| 3 | The saved bundle supports raw-row segment assignment. | The segmentation is reusable for new adult prospects, not just a notebook visualization. |